# Stress Prediction v13 LB Fix

This notebook replaces the over-prior-calibrated long-window model that scored poorly on LB. It uses compact physiological features, short + medium windows, stratified final ensembling, and writes several calibration candidates. Recommended default: `submission.csv` with `alpha=1.0`.

In [1]:
%pip -q install lightgbm scikit-learn pandas numpy scipy



[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats as spstats

from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight

import lightgbm as lgb

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path('.')
TRAIN_DATA  = pd.read_csv(DATA_DIR / 'train-sensor.csv')
TRAIN_LABEL = pd.read_csv(DATA_DIR / 'train-label.csv')
TEST_DATA   = pd.read_csv(DATA_DIR / 'test-sensor.csv')
TEST_LABEL  = pd.read_csv(DATA_DIR / 'test-label.csv')

TRAIN_LABEL['timestamp'] = TRAIN_LABEL['timestamp'].astype(float)
TEST_LABEL['timestamp']  = TEST_LABEL['timestamp'].astype(float)
for df in [TRAIN_DATA, TEST_DATA]:
    df['timestamp'] = df['timestamp'].astype(float)

print('TRAIN_DATA :', TRAIN_DATA.shape)
print('TRAIN_LABEL:', TRAIN_LABEL.shape)
print('TEST_DATA  :', TEST_DATA.shape)
print('TEST_LABEL :', TEST_LABEL.shape)
print('\nStress distribution:')
print(TRAIN_LABEL['stress'].value_counts().sort_index())


TRAIN_DATA : (4694400, 8)
TRAIN_LABEL: (815, 4)
TEST_DATA  : (5921280, 8)
TEST_LABEL : (1028, 4)

Stress distribution:
stress
0.0    162
1.0     66
2.0    587
Name: count, dtype: int64


## Feature Extraction

Key changes versus the bad LB notebook: fewer generic features, stronger stress-specific deltas/HRV/EDA features, short 15s window from the improved baseline, and per-subject baseline-normalized features.

In [3]:
SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']
WINDOWS = {
    's15': 15_000,
    's60': 60_000,
    's180': 180_000,
}

STAT_NAMES = ['mean', 'std', 'min', 'max', 'median', 'q25', 'q75', 'iqr', 'range', 'first', 'last', 'delta', 'slope', 'mad']

def safe_stats(values, prefix):
    v = pd.Series(values).dropna().astype(float).values
    out = {}
    if len(v) == 0:
        for s in STAT_NAMES:
            out[f'{prefix}_{s}'] = np.nan
        return out
    q25, q75 = np.percentile(v, [25, 75])
    x = np.linspace(0.0, 1.0, len(v))
    out[f'{prefix}_mean'] = float(np.mean(v))
    out[f'{prefix}_std'] = float(np.std(v))
    out[f'{prefix}_min'] = float(np.min(v))
    out[f'{prefix}_max'] = float(np.max(v))
    out[f'{prefix}_median'] = float(np.median(v))
    out[f'{prefix}_q25'] = float(q25)
    out[f'{prefix}_q75'] = float(q75)
    out[f'{prefix}_iqr'] = float(q75 - q25)
    out[f'{prefix}_range'] = float(np.max(v) - np.min(v))
    out[f'{prefix}_first'] = float(v[0])
    out[f'{prefix}_last'] = float(v[-1])
    out[f'{prefix}_delta'] = float(v[-1] - v[0])
    out[f'{prefix}_slope'] = float(np.polyfit(x, v, 1)[0]) if len(v) > 2 else 0.0
    out[f'{prefix}_mad'] = float(np.mean(np.abs(v - np.mean(v))))
    return out

def hrv_features(bpm_series, prefix):
    bpm = pd.Series(bpm_series).dropna().astype(float).values
    keys = ['sdnn', 'rmssd', 'pnn25', 'pnn50', 'mean_rr', 'cv_rr']
    if len(bpm) < 10:
        return {f'{prefix}_{k}': np.nan for k in keys}
    # heart_rate is repeated at sensor frequency; downsample to reduce duplicated RR values.
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    diff = np.diff(rr)
    sdnn = float(np.std(rr))
    mean_rr = float(np.mean(rr))
    return {
        f'{prefix}_sdnn': sdnn,
        f'{prefix}_rmssd': float(np.sqrt(np.mean(diff ** 2))) if len(diff) else 0.0,
        f'{prefix}_pnn25': float(np.mean(np.abs(diff) > 25)) if len(diff) else 0.0,
        f'{prefix}_pnn50': float(np.mean(np.abs(diff) > 50)) if len(diff) else 0.0,
        f'{prefix}_mean_rr': mean_rr,
        f'{prefix}_cv_rr': sdnn / mean_rr if mean_rr > 1e-6 else 0.0,
    }

def add_label_time_features(labels):
    lab = labels.copy().sort_values(['pid', 'timestamp']).reset_index(drop=True)
    extra = {}
    for pid, g in lab.groupby('pid', sort=False):
        ts = g['timestamp'].values.astype(float)
        gaps = np.r_[np.nan, np.diff(ts)]
        session = np.cumsum(np.r_[0, np.diff(ts) > 30 * 60 * 1000])
        pos = pd.Series(session).groupby(session).cumcount().values
        session_start_ts = pd.Series(ts).groupby(session).transform('first').values
        for idx, lid in enumerate(g['id'].values):
            extra[lid] = {
                'label_gap_prev_ms': gaps[idx],
                'label_session_id': session[idx],
                'label_pos_in_session': pos[idx],
                'label_since_session_start_ms': ts[idx] - session_start_ts[idx],
                'label_order_frac_pid': idx / max(len(g) - 1, 1),
                'label_hour': ((ts[idx] / 1000) / 3600) % 24,
            }
    return extra

def extract_features(label_df, sensor_df):
    label_time = add_label_time_features(label_df)
    sensor = sensor_df.copy()
    for c in SENSOR_COLS:
        sensor[c] = pd.to_numeric(sensor[c], errors='coerce')
    sensor['accel_mag'] = np.sqrt(sensor['accel_x'] ** 2 + sensor['accel_y'] ** 2 + sensor['accel_z'] ** 2)
    all_cols = SENSOR_COLS + ['accel_mag']
    by_pid = {pid: g.sort_values('timestamp').reset_index(drop=True) for pid, g in sensor.groupby('pid')}

    rows = []
    for n, row in enumerate(label_df.itertuples(index=False), 1):
        lid = int(row.id)
        pid = row.pid
        ts = float(row.timestamp)
        sg = by_pid.get(pid)
        feat = {'id': lid, **label_time.get(lid, {})}
        if sg is None:
            rows.append(feat)
            continue

        t = sg['timestamp'].values
        base_mean = sg[all_cols].mean()
        base_std = sg[all_cols].std(ddof=0).replace(0, np.nan)

        for wname, wms in WINDOWS.items():
            win = sg[(t >= ts - wms) & (t <= ts)]
            feat[f'{wname}_count'] = len(win)
            feat[f'{wname}_span_ms'] = float(win['timestamp'].max() - win['timestamp'].min()) if len(win) else 0.0
            for col in all_cols:
                feat.update(safe_stats(win[col], f'{wname}_{col}'))
                # Within-subject normalization: keeps physiology relative to that nurse's own baseline.
                z = (win[col] - base_mean[col]) / base_std[col] if len(win) else []
                feat.update(safe_stats(z, f'{wname}_{col}_z'))

            if len(win):
                feat.update(hrv_features(win['heart_rate'], f'{wname}_hrv'))
                eda = win['eda'].dropna().astype(float).values
                hr = win['heart_rate'].dropna().astype(float).values
                feat[f'{wname}_eda_absdiff_mean'] = float(np.mean(np.abs(np.diff(eda)))) if len(eda) > 1 else 0.0
                feat[f'{wname}_eda_rise_rate'] = float(np.mean(np.diff(eda) > 0)) if len(eda) > 1 else 0.0
                feat[f'{wname}_hr_absdiff_mean'] = float(np.mean(np.abs(np.diff(hr)))) if len(hr) > 1 else 0.0
            else:
                for k in ['eda_absdiff_mean', 'eda_rise_rate', 'hr_absdiff_mean']:
                    feat[f'{wname}_{k}'] = np.nan
                feat.update({f'{wname}_hrv_{k}': np.nan for k in ['sdnn', 'rmssd', 'pnn25', 'pnn50', 'mean_rr', 'cv_rr']})

        rows.append(feat)
        if n % 200 == 0:
            print(f'  extracted {n}/{len(label_df)}')
    return pd.DataFrame(rows).set_index('id')

print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA)
print('train features:', train_features.shape)
print('Extracting test features...')
test_features = extract_features(TEST_LABEL, TEST_DATA)
print('test features:', test_features.shape)


Extracting train features...
  extracted 200/815
  extracted 400/815
  extracted 600/815
  extracted 800/815
train features: (815, 627)
Extracting test features...
  extracted 200/1028
  extracted 400/1028
  extracted 600/1028
  extracted 800/1028
  extracted 1000/1028
test features: (1028, 627)


In [4]:
y = TRAIN_LABEL.set_index('id').loc[train_features.index, 'stress'].astype(int)
groups = TRAIN_LABEL.set_index('id').loc[train_features.index, 'pid'].astype(str)

feature_cols = sorted(set(train_features.columns) | set(test_features.columns))
X = train_features.reindex(columns=feature_cols)
X_test = test_features.reindex(columns=feature_cols)

imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

# Remove constant columns after imputation.
constant_cols = [c for c in X_imp.columns if X_imp[c].nunique(dropna=False) <= 1]
X_imp = X_imp.drop(columns=constant_cols)
X_test_imp = X_test_imp.drop(columns=constant_cols)

counts = Counter(y)
total = len(y)
train_prior = np.array([counts[i] / total for i in range(3)])
# Capping class 1 avoids the model turning the rare class into noise, while still helping BA.
class_weights = {0: total / (3 * counts[0]), 1: min(total / (3 * counts[1]), 2.5), 2: total / (3 * counts[2])}
sample_weights = np.array([class_weights[int(v)] for v in y])

print('Final matrix:', X_imp.shape, X_test_imp.shape)
print('Class weights:', {k: round(v, 3) for k, v in class_weights.items()})
print('Train prior:', {i: round(train_prior[i], 3) for i in range(3)})


Final matrix: (815, 621) (1028, 621)
Class weights: {0: 1.677, 1: 2.5, 2: 0.463}
Train prior: {0: np.float64(0.199), 1: np.float64(0.081), 2: np.float64(0.72)}


## Honest Check: Leave One Subject Out

This is not used to tune thresholds directly, but it catches the most dangerous subject leakage. Do not expect this number to match Kaggle exactly.

In [5]:
LGBM_PARAMS = dict(
    n_estimators=1200,
    learning_rate=0.018,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=6,
    subsample=0.70,
    colsample_bytree=0.65,
    reg_alpha=0.4,
    reg_lambda=0.6,
    class_weight='balanced',
    objective='multiclass',
    num_class=3,
    n_jobs=-1,
    verbose=-1,
)

print('=== Leave-One-PID-Out sanity check ===')
logo_scores = []
for tr_idx, va_idx in LeaveOneGroupOut().split(X_imp, y, groups):
    pid = groups.iloc[va_idx[0]]
    y_va = y.iloc[va_idx]
    if y_va.nunique() < 2:
        print(f'  skip {pid}: validation has one class only')
        continue
    model = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': RANDOM_SEED})
    model.fit(
        X_imp.iloc[tr_idx], y.iloc[tr_idx],
        sample_weight=sample_weights[tr_idx],
        eval_set=[(X_imp.iloc[va_idx], y_va)],
        callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(-1)],
    )
    pred = model.predict(X_imp.iloc[va_idx])
    score = balanced_accuracy_score(y_va, pred)
    logo_scores.append(score)
    print(f'  {pid}: BA={score:.4f} true={dict(Counter(y_va))} pred={dict(Counter(pred))}')
print(f'LOPO mean BA: {np.mean(logo_scores):.4f} +/- {np.std(logo_scores):.4f}')


=== Leave-One-PID-Out sanity check ===
  43JW: BA=0.1374 true={2: 91, 0: 2} pred={np.int64(0): 61, np.int64(1): 5, np.int64(2): 27}
  C8Q6: BA=0.4894 true={2: 142, 0: 10} pred={np.int64(2): 149, np.int64(1): 3}
  DT5C: BA=0.0632 true={0: 58, 2: 18, 1: 14} pred={np.int64(1): 63, np.int64(0): 27}
  F1ZM: BA=0.4776 true={2: 134, 1: 3} pred={np.int64(0): 4, np.int64(2): 131, np.int64(1): 2}
  HDS9: BA=0.4103 true={0: 18, 2: 117} pred={np.int64(2): 114, np.int64(0): 21}
  P4DZ: BA=0.3333 true={1: 49, 0: 53, 2: 42} pred={np.int64(1): 144}
  TPQI: BA=0.5161 true={2: 43, 0: 21} pred={np.int64(0): 22, np.int64(2): 36, np.int64(1): 6}
LOPO mean BA: 0.3468 +/- 0.1667


## Final Ensemble

The final models use stratified folds, like the stronger prior notebooks. This is intentionally different from the bad LB run, whose group-fold final ensemble was too weak and too prior-calibrated.

In [6]:
SEEDS = [42, 7, 123, 2026, 77]
raw_test_probas = []
oof = np.zeros((len(X_imp), 3))

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    seed_test = np.zeros((len(X_test_imp), 3))
    seed_scores = []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_imp, y), 1):
        model = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        model.fit(
            X_imp.iloc[tr_idx], y.iloc[tr_idx],
            sample_weight=sample_weights[tr_idx],
            eval_set=[(X_imp.iloc[va_idx], y.iloc[va_idx])],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)],
        )
        va_proba = model.predict_proba(X_imp.iloc[va_idx])
        te_proba = model.predict_proba(X_test_imp)
        oof[va_idx] += va_proba / len(SEEDS)
        seed_test += te_proba / 5
        score = balanced_accuracy_score(y.iloc[va_idx], np.argmax(va_proba, axis=1))
        seed_scores.append(score)
        print(f'Seed {seed} fold {fold}: BA={score:.4f}')
    raw_test_probas.append(seed_test)
    print(f'Seed {seed} mean BA={np.mean(seed_scores):.4f}')

raw_proba = np.mean(raw_test_probas, axis=0)
oof_ba = balanced_accuracy_score(y, np.argmax(oof, axis=1))
print('\nOOF BA from stratified ensemble:', round(oof_ba, 5))
print('Raw test distribution:', dict(Counter(np.argmax(raw_proba, axis=1))))


Seed 42 fold 1: BA=0.8260
Seed 42 fold 2: BA=0.8998
Seed 42 fold 3: BA=0.9529
Seed 42 fold 4: BA=0.9298
Seed 42 fold 5: BA=0.9006
Seed 42 mean BA=0.9018
Seed 7 fold 1: BA=0.9317
Seed 7 fold 2: BA=0.8467
Seed 7 fold 3: BA=0.9611
Seed 7 fold 4: BA=0.9375
Seed 7 fold 5: BA=0.8862
Seed 7 mean BA=0.9126
Seed 123 fold 1: BA=0.9337
Seed 123 fold 2: BA=0.7969
Seed 123 fold 3: BA=0.9182
Seed 123 fold 4: BA=0.9442
Seed 123 fold 5: BA=0.8397
Seed 123 mean BA=0.8865
Seed 2026 fold 1: BA=0.8089
Seed 2026 fold 2: BA=0.9495
Seed 2026 fold 3: BA=0.9109
Seed 2026 fold 4: BA=0.9290
Seed 2026 fold 5: BA=0.8501
Seed 2026 mean BA=0.8897
Seed 77 fold 1: BA=0.9080
Seed 77 fold 2: BA=0.9026
Seed 77 fold 3: BA=0.8969
Seed 77 fold 4: BA=0.8750
Seed 77 fold 5: BA=0.9222
Seed 77 mean BA=0.9009

OOF BA from stratified ensemble: 0.90375
Raw test distribution: {np.int64(2): 824, np.int64(0): 186, np.int64(1): 18}


In [7]:
# Calibration sweep.
# alpha=0.0 is raw probabilities. Higher alpha pushes predictions toward the train prior.
# The previous bad LB result was already close to train-prior/class-2-heavy, so the default here is alpha=1.0,
# not 1.5 or 1.8.
ALPHAS = [0.0, 0.6, 0.8, 1.0, 1.2, 1.5]
RECOMMENDED_ALPHA = 1.0
out_dir = Path('submission_alpha_sweep2')
out_dir.mkdir(exist_ok=True)

for alpha in ALPHAS:
    if alpha == 0.0:
        proba = raw_proba.copy()
    else:
        proba = raw_proba * (train_prior ** alpha)
        proba = proba / proba.sum(axis=1, keepdims=True)
    pred = np.argmax(proba, axis=1).astype(int)
    sub = pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': pred})
    fname = out_dir / f'submission_alpha_{str(alpha).replace(".", "p")}.csv'
    sub.to_csv(fname, index=False)
    print(f'alpha={alpha:>3}: {dict(Counter(pred))} -> {fname}')
    if alpha == RECOMMENDED_ALPHA:
        sub.to_csv('submissionv13.csv', index=False)
        final_preds = pred

print('\nRecommended file: submission.csv')
print('Recommended distribution:', dict(Counter(final_preds)))
print(pd.read_csv('submission.csv').head(10))


alpha=0.0: {np.int64(2): 824, np.int64(0): 186, np.int64(1): 18} -> submission_alpha_sweep2/submission_alpha_0p0.csv
alpha=0.6: {np.int64(2): 909, np.int64(0): 107, np.int64(1): 12} -> submission_alpha_sweep2/submission_alpha_0p6.csv
alpha=0.8: {np.int64(2): 932, np.int64(0): 90, np.int64(1): 6} -> submission_alpha_sweep2/submission_alpha_0p8.csv
alpha=1.0: {np.int64(2): 941, np.int64(0): 83, np.int64(1): 4} -> submission_alpha_sweep2/submission_alpha_1p0.csv
alpha=1.2: {np.int64(2): 960, np.int64(0): 67, np.int64(1): 1} -> submission_alpha_sweep2/submission_alpha_1p2.csv
alpha=1.5: {np.int64(2): 976, np.int64(0): 52} -> submission_alpha_sweep2/submission_alpha_1p5.csv

Recommended file: submission.csv
Recommended distribution: {np.int64(2): 941, np.int64(0): 83, np.int64(1): 4}
     id  stress
0  1227       2
1  1228       2
2  1229       2
3  1230       2
4  1231       2
5  1232       2
6  1233       2
7  1234       2
8  1235       2
9  1236       2
